# Experimento 01: Regresión Logística (Estudio SMOTE Dual)

En este notebook ejecutamos el segundo nivel de complejidad algorítmica: un modelo lineal probabilístico (**Logistic Regression**).

El objetivo de este experimento es doble:
1.  **Benchmarking Algorítmico:** Superar el rendimiento del modelo Naive Bayes (Baseline) utilizando la misma estructura de datos desbalanceada.
2.  **Estudio de Ablación SMOTE:** Demostrar científicamente que la inyección de datos sintéticos (*Synthetic Minority Over-sampling Technique*) es obligatoria para recuperar la métrica Recall en clases minoritarias críticas (*Service Outages*).

Todas las pruebas se ejecutarán en paralelo para el idioma Inglés y Español.

### Paso 1: Carga Express de Matrices (Arquitectura Desacoplada)

In [1]:
import scipy.sparse
import pandas as pd
import time

print("Iniciando carga de matrices en memoria (Modo MLOps)...")
start_load = time.time()

# 1. CARGA DEL DATASET INGLÉS (Matrices matemáticas puras)
# Usamos load_npz para descomprimir el formato de SciPy instantáneamente
en_X_train = scipy.sparse.load_npz("../data/features/en_X_train_tfidf.npz")
en_X_val = scipy.sparse.load_npz("../data/features/en_X_val_tfidf.npz")
en_X_test = scipy.sparse.load_npz("../data/features/en_X_test_tfidf.npz")

# Cargamos las etiquetas objetivo. 
# Usamos .iloc[:, 0] para forzar a que Pandas lo lea como una lista unidimensional plana (Serie)
# Si no ponemos .iloc[:, 0], Scikit-Learn se quejará después diciendo que le estás pasando una tabla de 2D en vez de 1D.
en_y_train = pd.read_csv("../data/features/en_y_train.csv").iloc[:, 0]
en_y_val = pd.read_csv("../data/features/en_y_val.csv").iloc[:, 0]
en_y_test = pd.read_csv("../data/features/en_y_test.csv").iloc[:, 0]

# 2. CARGA DEL DATASET ESPAÑOL (Misma lógica)
es_X_train = scipy.sparse.load_npz("../data/features/es_X_train_tfidf.npz")
es_X_val = scipy.sparse.load_npz("../data/features/es_X_val_tfidf.npz")
es_X_test = scipy.sparse.load_npz("../data/features/es_X_test_tfidf.npz")

es_y_train = pd.read_csv("../data/features/es_y_train.csv").iloc[:, 0]
es_y_val = pd.read_csv("../data/features/es_y_val.csv").iloc[:, 0]
es_y_test = pd.read_csv("../data/features/es_y_test.csv").iloc[:, 0]

end_load = time.time()
print(f"✅ Todos los datasets cargados en {round(end_load - start_load, 2)} segundos.")

# 3. AUDITORÍA DE DIMENSIONES (Control de Calidad)
# Nos aseguramos de que las dimensiones cruzadas coinciden perfectamente
print("\n--- AUDITORÍA DE DIMENSIONES ---")
print(f"Inglés  - X_Train: {en_X_train.shape} -> Y_Train: {en_y_train.shape}")
print(f"Español - X_Train: {es_X_train.shape} -> Y_Train: {es_y_train.shape}")

Iniciando carga de matrices en memoria (Modo MLOps)...
✅ Todos los datasets cargados en 0.14 segundos.

--- AUDITORÍA DE DIMENSIONES ---
Inglés  - X_Train: (16181, 10000) -> Y_Train: (16181,)
Español - X_Train: (16706, 10000) -> Y_Train: (16706,)


### Paso 2: Evaluación Control (LogReg Sin SMOTE)

Antes de alterar los datos artificialmente, establecemos el primer punto de control del Estudio de Ablación. Entrenamos un modelo de Regresión Logística (modelo lineal probabilístico) sobre la matriz fuertemente desbalanceada.

Esto nos permitirá comprobar empíricamente si la simple evolución del algoritmo (de Naive Bayes a LogReg) es suficiente para rescatar la clase minoritaria *Service Outages and Maintenance*, o si el colapso del Recall es un problema de estructura de datos que exige SMOTE.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
import time

# Función Fábrica actualizada para imprimir el reporte completo y sin avisos
def train_evaluate_logreg(X_train, y_train, X_val, y_val, exp_name):
    print(f"\n=======================================================")
    print(f"--- ENTRENANDO REGRESIÓN LOGÍSTICA - {exp_name} ---")
    
    # 1. INSTANCIACIÓN (Sin n_jobs=-1 para evitar el FutureWarning)
    model = LogisticRegression(max_iter=1000)
    
    # 2. ENTRENAMIENTO
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = round(time.time() - start_train, 4)
    
    # 3. INFERENCIA
    start_inf = time.time()
    y_pred = model.predict(X_val)
    inf_time_ms = round((time.time() - start_inf) * 1000, 2)
    
    # 4. REPORTE COMPLETO
    print(f"[T. Entrenamiento: {train_time} seg | T. Inferencia: {inf_time_ms} ms]\n")
    print(classification_report(y_val, y_pred, zero_division=0))
    
    # 5. RETORNO DE VARIABLES (Para el Tracker)
    f1_macro = round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4)
    report_dict = classification_report(y_val, y_pred, output_dict=True, zero_division=0)
    f1_minority = round(report_dict.get('Service Outages and Maintenance', {}).get('f1-score', 0), 4)
    
    return train_time, inf_time_ms, f1_macro, f1_minority

# EJECUTAMOS CONTROL (SIN SMOTE)
en_train_time_none, en_inf_time_none, en_f1_none, en_f1_min_none = train_evaluate_logreg(
    en_X_train, en_y_train, en_X_val, en_y_val, "INGLÉS (SIN SMOTE)"
)

es_train_time_none, es_inf_time_none, es_f1_none, es_f1_min_none = train_evaluate_logreg(
    es_X_train, es_y_train, es_X_val, es_y_val, "ESPAÑOL (SIN SMOTE)"
)


--- ENTRENANDO REGRESIÓN LOGÍSTICA - INGLÉS (SIN SMOTE) ---
[T. Entrenamiento: 2.3864 seg | T. Inferencia: 2.48 ms]

                                 precision    recall  f1-score   support

           Billing and Payments       0.89      0.73      0.80       381
               Customer Service       0.44      0.38      0.41       569
                     IT Support       0.42      0.18      0.25       451
                Product Support       0.43      0.39      0.41       701
            Sales and Pre-Sales       0.92      0.10      0.19       115
Service Outages and Maintenance       0.81      0.43      0.56       149
              Technical Support       0.49      0.77      0.60      1102

                       accuracy                           0.51      3468
                      macro avg       0.63      0.43      0.46      3468
                   weighted avg       0.53      0.51      0.49      3468


--- ENTRENANDO REGRESIÓN LOGÍSTICA - ESPAÑOL (SIN SMOTE) ---
[T. Entrenamie

### Paso 3: Estudio de Ablación (Inyección Sintética con SMOTE)

Para corregir la ceguera algorítmica ante las averías críticas (*Service Outages*), aplicamos **SMOTE** (Synthetic Minority Over-sampling Technique). 

A diferencia de duplicar filas (lo cual genera sobreajuste), SMOTE opera en el espacio vectorial TF-IDF de 10.000 dimensiones. Interpola geométricamente la distancia entre los tickets de averías existentes y crea nuevos vectores (tickets falsos) a lo largo de esas líneas. 

El objetivo es equiparar el volumen de todas las colas de soporte a la clase mayoritaria (*Technical Support*) estrictamente en el conjunto de Entrenamiento, sin contaminar el conjunto de Validación.

In [3]:
from imblearn.over_sampling import SMOTE
import time

print("--- INICIANDO INYECCIÓN SINTÉTICA (SMOTE) ---")

# 1. INSTANCIACIÓN DE SMOTE
smote = SMOTE(random_state=42)

# 2. TRANSFORMACIÓN GEOMÉTRICA (INGLÉS)
print("\nAplicando SMOTE al dataset en Inglés...")
start_smote_en = time.time()
en_X_train_smote, en_y_train_smote = smote.fit_resample(en_X_train, en_y_train)
print(f"✅ Inglés balanceado en {round(time.time() - start_smote_en, 2)} seg.")

# 3. TRANSFORMACIÓN GEOMÉTRICA (ESPAÑOL)
print("Aplicando SMOTE al dataset en Español...")
start_smote_es = time.time()
es_X_train_smote, es_y_train_smote = smote.fit_resample(es_X_train, es_y_train)
print(f"✅ Español balanceado en {round(time.time() - start_smote_es, 2)} seg.")

# 4. AUDITORÍA VISUAL (El Impacto en Negocio)
print("\n--- AUDITORÍA ANTES VS DESPUÉS (Volumen de Averías Críticas) ---")
print("INGLÉS:")
print(f" Antes  : {sum(en_y_train == 'Service Outages and Maintenance')} tickets")
print(f" Después: {sum(en_y_train_smote == 'Service Outages and Maintenance')} tickets")

# 5. RE-ENTRENAMIENTO (Test de Fuego con el reporte completo)
# Al llamar a la función que modificamos antes, ahora te imprimirá las tablas enteras
en_train_time_smote, en_inf_time_smote, en_f1_smote, en_f1_min_smote = train_evaluate_logreg(
    en_X_train_smote, en_y_train_smote, en_X_val, en_y_val, "INGLÉS (CON SMOTE)"
)

es_train_time_smote, es_inf_time_smote, es_f1_smote, es_f1_min_smote = train_evaluate_logreg(
    es_X_train_smote, es_y_train_smote, es_X_val, es_y_val, "ESPAÑOL (CON SMOTE)"
)

--- INICIANDO INYECCIÓN SINTÉTICA (SMOTE) ---

Aplicando SMOTE al dataset en Inglés...
✅ Inglés balanceado en 0.59 seg.
Aplicando SMOTE al dataset en Español...
✅ Español balanceado en 0.61 seg.

--- AUDITORÍA ANTES VS DESPUÉS (Volumen de Averías Críticas) ---
INGLÉS:
 Antes  : 695 tickets
 Después: 5139 tickets

--- ENTRENANDO REGRESIÓN LOGÍSTICA - INGLÉS (CON SMOTE) ---
[T. Entrenamiento: 5.9526 seg | T. Inferencia: 1.73 ms]

                                 precision    recall  f1-score   support

           Billing and Payments       0.76      0.78      0.77       381
               Customer Service       0.43      0.45      0.44       569
                     IT Support       0.36      0.43      0.39       451
                Product Support       0.47      0.43      0.45       701
            Sales and Pre-Sales       0.32      0.54      0.40       115
Service Outages and Maintenance       0.50      0.64      0.56       149
              Technical Support       0.59      0.50    

### Paso 4: Consolidación MLOps (Registro Central de Experimentos)

Para mantener la trazabilidad absoluta del TFM, procedemos a volcar los 4 nuevos modelos entrenados (2 en Inglés, 2 en Español) en nuestros archivos maestros de seguimiento (`tracker_en.csv` y `tracker_es.csv`). 

Esto nos permite construir un histórico permanente de métricas. Hacemos uso explícito de la nueva columna `hyperparameters` que introdujimos en la fase anterior para documentar que la Regresión Logística se ejecutó con `max_iter=1000`.

In [4]:
import pandas as pd

print("--- GUARDANDO EXPERIMENTOS EN EL TRACKER CENTRAL ---")

# 1. CARGAMOS LOS TRACKERS FÍSICOS DESDE EL DISCO DURO
tracker_en = pd.read_csv("../data/processed/tracker_en.csv")
tracker_es = pd.read_csv("../data/processed/tracker_es.csv")

# 2. EMPAQUETAMOS LOS NUEVOS EXPERIMENTOS (INGLÉS)
nuevos_experimentos_en = [
    {
        'exp_id': 'EN_L1_TFIDF_LOGREG_NONE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'logreg',
        'balancing': 'none',
        'hyperparameters': 'max_iter=1000',
        'train_time_sec': en_train_time_none,
        'inference_time_ms': en_inf_time_none,
        'f1_macro': en_f1_none,
        'f1_minority_class': en_f1_min_none
    },
    {
        'exp_id': 'EN_L1_TFIDF_LOGREG_SMOTE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'logreg',
        'balancing': 'smote',
        'hyperparameters': 'max_iter=1000',
        'train_time_sec': en_train_time_smote,
        'inference_time_ms': en_inf_time_smote,
        'f1_macro': en_f1_smote,
        'f1_minority_class': en_f1_min_smote
    }
]

# 3. EMPAQUETAMOS LOS NUEVOS EXPERIMENTOS (ESPAÑOL)
nuevos_experimentos_es = [
    {
        'exp_id': 'ES_L1_TFIDF_LOGREG_NONE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'logreg',
        'balancing': 'none',
        'hyperparameters': 'max_iter=1000',
        'train_time_sec': es_train_time_none,
        'inference_time_ms': es_inf_time_none,
        'f1_macro': es_f1_none,
        'f1_minority_class': es_f1_min_none
    },
    {
        'exp_id': 'ES_L1_TFIDF_LOGREG_SMOTE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'logreg',
        'balancing': 'smote',
        'hyperparameters': 'max_iter=1000',
        'train_time_sec': es_train_time_smote,
        'inference_time_ms': es_inf_time_smote,
        'f1_macro': es_f1_smote,
        'f1_minority_class': es_f1_min_smote
    }
]

# 4. INYECTAMOS Y SOBREESCRIBIMOS (INGLÉS)
tracker_en = pd.concat([tracker_en, pd.DataFrame(nuevos_experimentos_en)], ignore_index=True)
tracker_en.to_csv("../data/processed/tracker_en.csv", index=False)

# 5. INYECTAMOS Y SOBREESCRIBIMOS (ESPAÑOL)
tracker_es = pd.concat([tracker_es, pd.DataFrame(nuevos_experimentos_es)], ignore_index=True)
tracker_es.to_csv("../data/processed/tracker_es.csv", index=False)

print("✅ Todos los experimentos han sido registrados con éxito.")

# 6. AUDITORÍA VISUAL DEL PROGRESO
print("\n--- TRACKER MAESTRO ACTUALIZADO (INGLÉS) ---")
display(tracker_en)
print("\n--- TRACKER MAESTRO ACTUALIZADO (ESPAÑOL) ---")
display(tracker_es)

--- GUARDANDO EXPERIMENTOS EN EL TRACKER CENTRAL ---
✅ Todos los experimentos han sido registrados con éxito.

--- TRACKER MAESTRO ACTUALIZADO (INGLÉS) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class,hyperparameters
0,EN_L1_TFIDF_MNB_NONE,queue,tfidf,mnb,none,0.0475,2.61,0.3347,0.2775,baseline default
1,EN_L1_TFIDF_LOGREG_NONE,queue,tfidf,logreg,none,2.3864,2.48,0.4598,0.5614,max_iter=1000
2,EN_L1_TFIDF_LOGREG_SMOTE,queue,tfidf,logreg,smote,5.9526,1.73,0.5079,0.5605,max_iter=1000



--- TRACKER MAESTRO ACTUALIZADO (ESPAÑOL) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class,hyperparameters
0,ES_L1_TFIDF_MNB_NONE,queue,tfidf,mnb,none,0.0425,2.42,0.3088,0.0870,baseline default
1,ES_L1_TFIDF_LOGREG_NONE,queue,tfidf,logreg,none,2.6192,1.92,0.4522,0.5066,max_iter=1000
2,ES_L1_TFIDF_LOGREG_SMOTE,queue,tfidf,logreg,smote,5.7504,2.17,0.5007,0.5938,max_iter=1000
